In [68]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

In [69]:
generator_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-8b")

In [70]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal['approved','needs_improvement'] = Field(...,description="Final Evaluation")
    feedback: str = Field(...,description="feedback for the tweet")

In [71]:
structred_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [72]:
class TweetState(TypedDict):

    topic:str
    tweet:str
    evaluation:Literal["approved","needs_improvement"]
    feedback:str
    iteration:str
    max_iteration:int

In [73]:
def generate_tweet(state:TweetState):

    #prompt
    messages = [
        SystemMessage(content="You are funny and clever twitter/X influencer."),
        HumanMessage(content=f"""
        Write a short, original, and hilarious tweet on the topic {state['topic']}.

        RULES:
        - Do not use question-answer format.
        - Max 280 Characters.
        - Use observational humour, irony, sarcasm, or cultural refrences.
        - Think in meme logic, punchlines, or relatable takes.
        - Use simple day to day english
        - This is version {state['iteration']+1}
""")
    ]

    #send to llm

    response = generator_llm.invoke(messages).content

    #update state
    return {'tweet':response}

In [74]:
def evaluate_tweet(state: TweetState):
    #prompt
    messages = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
        ),
        
        HumanMessage(
            content=f"""
Evaluate the following Tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Don't end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-uncle universe" or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining the strengths and weaknesses
"""
        )
    ]
    
    #llm
    response = structred_evaluator_llm.invoke(messages)
    
    # --- LOOP TESTING BLOCK ---
    # If this is the first iteration, ignore the LLM's praise and force a rejection
    if state.get('iteration', 1) == 1:
        print("\n🛠️ DEBUG: Forcing 'needs_improvement' on iteration 1 to test the loop! 🛠️\n")
        return {
            'evaluation': 'needs_improvement', 
            'feedback': 'TEST FEEDBACK: This is way too polite and safe. Inject some biting sarcasm and make the punchline hit harder.'
        }
    # --------------------------

    return {'evaluation': response.evaluation, 'feedback': response.feedback}

In [75]:
def optimize_tweet(state:TweetState):

    messages = [
        SystemMessage(
            content="You punch up tweets for virality and humor based on given feedback."
        ),
        HumanMessage(
            content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"

Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
"""
        )
    ]

    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration']+1

    return {"tweet": response,'iteration':iteration}



In [76]:
def route_eval(state:TweetState):

    if state['evaluation']=='approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [77]:
graph = StateGraph(TweetState)

#nodes
graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)


#edges
graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')

graph.add_conditional_edges('evaluate',route_eval,{'approved':END, 'needs_improvement':'optimize'})
graph.add_edge('optimize','evaluate')

workflow = graph.compile()



In [78]:
initial_state = {
    'topic':'Indian_railways',
    'iteration' : 1,
    'max_iteration':5
}

workflow.invoke(initial_state)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 17.480174385s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url